# 2. Train and test the model

**What you will learn**

- How the model is tested year by year on years it never saw ("rolling origin"), and why one year is left out between training and test.
- How the two main engines, the logit and gradient boosting, are fitted by the package's own code (not by new code written here).
- What ROC-AUC and PR-AUC mean in plain words, with a small example you can check by hand.

**What you need**

- Notebook 1 (or at least the idea of the panel, the label and the inputs).
- Python with the packages in `model/requirements.txt` and `jupyter`. No licensed data and no internet.
- About a minute to run (47 to 69 seconds on one 16-core laptop; the full year-by-year test runs as a command inside the notebook). The temporary example data is deleted in the last cell.

> **The example data is made up.** Every company here is called "Example Co ...", and every number comes from a random
> number generator. A result on this data says nothing about real companies. The example is only there so that all the
> code runs on your own computer. Do not compare its numbers with the lecture's real results as if they measured the same thing.

## Set up

As in notebook 1: find the code, make a temporary folder, and write an example table into it.

In [ ]:
import os, sys, shutil, subprocess, tempfile
from pathlib import Path

import numpy as np
import pandas as pd

# Find the repository folder (the one that holds model/src/tcf_credit). This works whether the notebook
# is started from the repository folder or from model/notebooks.
here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "model" / "src" / "tcf_credit").is_dir())
SRC, SYN = root / "model" / "src", root / "model" / "synthetic"
sys.path[:0] = [str(SRC), str(SYN)]
# Commands that we start from the notebook need the package on their path too.
os.environ["PYTHONPATH"] = os.pathsep.join([str(SRC)] + ([os.environ["PYTHONPATH"]] if os.environ.get("PYTHONPATH") else []))

work = Path(tempfile.mkdtemp(prefix="tcf_nb2_"))   # a temporary folder; the last cell deletes it
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)
print("repository:", root)

In [ ]:
panel_path = work / "example_panel.csv.gz"
done = subprocess.run([sys.executable, str(SYN / "make_panel.py"), "--out", str(panel_path), "--firms", "400", "--seed", "1"],
                      check=True, capture_output=True, text=True)
print(done.stdout.strip())

## Rolling-origin testing

A model is only useful if it works on companies and years it has **not** seen. So we never test on the data we trained on.
We also respect time: in real life you only have the past. The test therefore walks forward through the years:

> For each test year **Y**: train on reports dated in years **up to Y-2**, and test on the reports dated in year **Y**.
> Year **Y-1** is left out on purpose.

("Origin" is the point in time where the training data stops. It rolls forward by one year each time.)

**Why leave out Y-1?** The label of a report looks 365 days ahead. A report filed in, say, November of Y-1 has a label that
is only decided by the end of year Y. When you score year Y in real life, that label is still unknown. If we trained on Y-1,
we would train on answers that were not yet known on the day. The cell below checks this on the example table.

In [ ]:
panel = pd.read_csv(panel_path)
panel["t"] = pd.to_datetime(panel["t"])
panel["year"] = panel["t"].dt.year
window_end = panel["t"] + pd.Timedelta(days=365)          # the day each report's 12-month label window ends

Y = 2020
start_of_Y = pd.Timestamp(f"{Y}-01-01")
for name, yr in [(f"reports of year Y-2 ({Y-2})", Y - 2), (f"reports of year Y-1 ({Y-1})", Y - 1)]:
    m = panel["year"] == yr
    share = (window_end[m] < start_of_Y).mean()
    print(f"{name}: label known before 1 January {Y} for {share:.0%} of them")

For year Y-2 every label is settled before the test year starts, so it is safe to train on. For year Y-1 almost none is: those labels are still open.
That is the one-year gap.

This is the plan for each test year, computed from the example table:

In [ ]:
df = panel[panel.has_prices == 1].copy()                  # the model uses market inputs, so it tests only reports that have them
rows = []
for Y in range(2016, 2025):
    tr, te = df[df.year <= Y - 2], df[df.year == Y]
    rows.append({"test year": Y, "train on years": f"up to {Y-2}", "left out": Y - 1,
                 "train reports": len(tr), "train downgrades": int(tr.label.sum()),
                 "test reports": len(te), "test downgrades": int(te.label.sum())})
pd.DataFrame(rows).set_index("test year")

## Fit the two main engines with the package's own code

The model's testing code is in `model/src/tcf_credit/evaluate.py`. We use its functions directly, so what you see here is
what the lecture's pipeline does:

- `Prep` prepares the inputs. Everything it learns (clipping limits, medians, scaling) comes from the **training rows only**. If it
  learned from the test rows, the test would be contaminated.
- `tune` chooses each model's setting (for example how hard the logit is held back from over-fitting) on validation years
  that end **before** the first test year. The settings are then frozen.
- `fit_predict` fits a model on the training rows and returns its probabilities.

We score each fit with **ROC-AUC** and **PR-AUC**, two quality numbers explained step by step further down in this notebook.

The two engines:

- **Logit** (logistic regression): a weighted sum of the inputs, turned into a probability between 0 and 1. It is the main model.
- **Gradient boosting**: many small decision trees, each one correcting the mistakes of the ones before. It can find
  curved and interacting patterns that a logit cannot.

In [ ]:
from tcf_credit import evaluate as E
from tcf_credit.features_v2 import V2_COLUMNS

# Prepare the table exactly as evaluate.main() does.
E.EXTRA[:] = V2_COLUMNS + E.MARKET
E.ONLY[:] = ["logit", "boosting"]
data = pd.read_csv(panel_path)
data = data[data.has_prices == 1].copy()
data["year"] = pd.to_datetime(data["t"]).dt.year
data["ebit_interest"] = data["ebit_interest"].clip(-50, 100)     # a tiny interest bill makes this ratio explode

params = E.tune(data, first=2016, end=2014)       # settings chosen on validation years 2012-2014, before any test year
print("chosen settings:", params["chosen"])

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

Y = 2020
train, test = data[data.year <= Y - 2], data[data.year == Y]
models = E.models(params)                          # name -> (how to build, inputs, impute, sectors)
hand = {}
for name, (make, cols_, impute, sectors) in models.items():
    _, p = E.fit_predict(make, cols_, impute, sectors, train, test)
    hand[name] = p
    print(f"{name:<9} test year {Y}: ROC-AUC {roc_auc_score(test.label, p):.3f}   PR-AUC {average_precision_score(test.label, p):.3f}")
print("share of downgrades in the test year (the PR-AUC of a model with no skill):", round(test.label.mean(), 3))

Next the whole year-by-year test, 2016 to 2024, run as a command exactly as you would from a terminal:

`python -m tcf_credit.evaluate PANEL OUT_DIR --first 2016 --last 2024`

It writes `pooled.csv` (all test years together), `per_year.csv` and `predictions.csv` (every test score) into the output folder.
This takes about a minute on a typical laptop.

In [ ]:
out_dir = work / "run"
res = subprocess.run([sys.executable, "-m", "tcf_credit.evaluate", str(panel_path), str(out_dir), "--first", "2016", "--last", "2024"],
                     capture_output=True, text=True)
if res.returncode != 0:
    print(res.stderr[-1500:])
assert res.returncode == 0
print("evaluate finished; files written:", sorted(p.name for p in out_dir.iterdir()))
pooled = pd.read_csv(out_dir / "pooled.csv")
per_year = pd.read_csv(out_dir / "per_year.csv")
print()
pooled[pooled.rule == "calibrated_cut"][["model", "n", "downgrades", "roc_auc", "pr_auc"]].set_index("model")

**How to read this table.** One row per model, all test years 2016-2024 pooled.

- `n`: the number of test reports. `downgrades`: how many of them were followed by an S&P downgrade within 12 months.
- `roc_auc`, `pr_auc`: the two quality numbers (explained in the next sections).
- The model names: `logit` and `boosting` (the two engines), `logit_rating_only` (a logit that sees only the S&P rating state), and
  `avg_logit_boosting` (the plain average of the first two scores).

`pooled.csv` has **two rows per model**, and a column `rule` that names the *flag rule* behind the counts in that row:

- `calibrated_cut`: a report is flagged if its score is above a line fixed from an **earlier** year's scores (the lecture's rule; notebook 03 explains it).
- `rank_top`: the riskiest 10% of each test year, by that year's own scores (a comparison only).

The two AUC columns are identical for both rules, because they use the scores and not the flags. We print the `calibrated_cut` rows.
The file has more columns: `base_rate` (share of reports that were downgraded), `brier` (the average squared gap between the probability and what happened;
lower is better), and the counts `flagged`, `TP`, `FP`, `FN`, `TN` and rates `miss_rate`, `false_alarm_rate`, `precision`, which are the subject of notebook 03.

A check that can fail: the hand-made fit above and the command must give the same test-year ROC-AUC for the same model.

In [ ]:
for name in hand:
    from_command = per_year[(per_year.model == name) & (per_year.year == Y) & (per_year.rule == "calibrated_cut")].auc.iloc[0]
    by_hand = roc_auc_score(test.label, hand[name])
    assert abs(from_command - by_hand) < 1e-6, (name, from_command, by_hand)
    print(f"{name:<9} year {Y}: command {from_command:.4f}  hand {by_hand:.4f}  same")

The table above also holds `logit_rating_only`, a logit that sees only the S&P rating state (rating, outlook, CreditWatch,
recent actions), and `avg_logit_boosting`, the plain average of the two scores. The rating-only line is the
"what you already know from the rating" benchmark. The question for the model is how much the other inputs add on top.

On the lecture's real data the answer was: rating state alone, ROC-AUC 0.757; logit with all inputs, 0.802.

## ROC-AUC and PR-AUC in plain words

The model gives each report a score between 0 and 1. How good are the scores? Two numbers are used.

**ROC-AUC** answers: *pick one company that was downgraded and one that was not, at random. How often does the model give the downgraded one the higher score?*
A coin flip gives 0.5. A perfect model gives 1.0. (AUC means "area under the curve"; the curve is the ROC curve. You do not need
the curve to understand the number.)

**PR-AUC** answers: *walk down the list from the highest score to the lowest. Each time you reach a company that was really downgraded, note what
share of the companies you have walked past are real downgrades (the "precision"). Average those shares.* A model with no skill
scores about the share of downgrades in the data (13.5% in the lecture's data), not 0.5. This number cares about the top of the list, where the
flags are, so it is the stricter test when downgrades are rare. (PR-AUC here is computed as "average precision", the usual way.)

A small example you can check by hand: ten companies, three of them downgraded.

In [ ]:
toy = pd.DataFrame({"company": list("ABCDEFGHIJ"),
                    "score":   [0.90, 0.80, 0.70, 0.60, 0.50, 0.40, 0.30, 0.20, 0.10, 0.05],
                    "downgraded": [1, 0, 1, 0, 0, 1, 0, 0, 0, 0]})
toy = toy.sort_values("score", ascending=False).reset_index(drop=True)

# ROC-AUC by hand: compare every (downgraded, not downgraded) pair.
up = toy[toy.downgraded == 1].score
down = toy[toy.downgraded == 0].score
wins = sum(a > b for a in up for b in down)
pairs = len(up) * len(down)
print(f"pairs: {len(up)} downgraded x {len(down)} not = {pairs}; the downgraded one scores higher in {wins}")
print(f"ROC-AUC by hand = {wins}/{pairs} = {wins / pairs:.4f}   sklearn says {roc_auc_score(toy.downgraded, toy.score):.4f}")
assert abs(wins / pairs - roc_auc_score(toy.downgraded, toy.score)) < 1e-12

# PR-AUC by hand: precision after each real downgrade, then the average.
toy["flagged so far"] = range(1, len(toy) + 1)
toy["downgrades so far"] = toy.downgraded.cumsum()
toy["precision so far"] = (toy["downgrades so far"] / toy["flagged so far"]).round(3)
hits = toy[toy.downgraded == 1]["precision so far"]
print(f"precision at each real downgrade: {list(hits)} -> average {hits.mean():.4f}   sklearn says {average_precision_score(toy.downgraded, toy.score):.4f}")
assert abs(hits.mean() - average_precision_score(toy.downgraded, toy.score)) < 1e-3
toy

Read the table: company A (rank 1) is a real downgrade, so precision is 1/1. Company C is the next real downgrade, at rank 3: 2 of 3
= 0.667. Company F, at rank 6: 3 of 6 = 0.5. The average of 1, 0.667 and 0.5 is 0.722. A no-skill model would have scored about 0.3 here (3 of 10 are downgrades).

Now the same two numbers for all the engines on the example data, next to the no-skill PR-AUC:

In [ ]:
base_rate = pooled[pooled.rule == "calibrated_cut"].base_rate.iloc[0]
summary = pooled[pooled.rule == "calibrated_cut"][["model", "roc_auc", "pr_auc"]].set_index("model")
summary.loc["(no skill)"] = [0.5, base_rate]
summary

These numbers are for made-up data and mean nothing about real companies. Use them only to check that you can read the table.

**What the lecture found on the real (licensed) data, for comparison.** Test years 2016-2024, 6,380 annual reports, of which
862 were followed by an S&P downgrade within 12 months (13.5%).

| Result (real data) | Value |
|---|---|
| Logit: ROC-AUC / PR-AUC | 0.802 / 0.494 |
| Other single engines (not the fly models): ROC-AUC | 0.787 to 0.795 (averages of several models, which include the logit: up to 0.809) |
| S&P rating state alone: ROC-AUC | 0.757 |
| Fly brain models: ROC-AUC (explained in notebook 04) | 0.759 to 0.807 (real wiring had no consistent edge over shuffled wiring) |
| Flag line (top 10% of an earlier year's scores; explained in notebook 03) | caught 407 of 862 downgrades, with 8.7% false alarms |

On the real data the logit (0.802) was within the range of the other non-fly models (0.787 to 0.809; the top of that range is the two averages of several models; the logit on the S&P rating state alone scored 0.757).
Each single engine scored slightly below the logit (0.787 to 0.795). The logit was named the main model on 4 November 2026, before the extra models (XGBoost, random forest, neural network, boosting with direction rules, averages and the fly models) were run. Its set of inputs was extended later, so the logit reported here is not exactly the one named that day. No single engine beat it. Only the logit-boosting average, which contains the logit, had an interval above zero (+0.007).

## Tidy up

The last cell deletes the temporary folder.

## Check yourself

1. We test on year 2020. Which years are used for training, and which year is left out? Why?

<details><summary>Answer</summary>

Training uses reports dated up to 2018. 2019 is left out: the labels of 2019 reports (a 12-month look-ahead) are not all known when the 2020 test starts,
so training on them would use answers that were not available on the day.
</details>

2. A model has ROC-AUC 0.80. What does that say, in one sentence?

<details><summary>Answer</summary>

If you pick one downgraded and one non-downgraded company at random, the model gives the downgraded one the higher score in about 80% of such pairs.
</details>

3. Only 13.5% of reports are followed by a downgrade. Why is a PR-AUC of 0.49 a good result, even though 0.49 is below 0.5?

<details><summary>Answer</summary>

PR-AUC is not compared with 0.5. A model with no skill scores about the share of downgrades, 0.135. So 0.49 is more than three times the no-skill level.
</details>

In [ ]:
shutil.rmtree(work, ignore_errors=True)
print("temporary folder deleted:", not work.exists())